In [2]:
!pip -q install datasets

import re
import numpy as np
import pandas as pd
from collections import Counter
from datasets import load_dataset

np.random.seed(42)

In [7]:
LANGUAGES = {
    "asm_Beng": "Assamese",
    "ben_Beng": "Bengali",
    "brx_Deva": "Bodo",
    "doi_Deva": "Dogri",
    "gom_Deva": "Konkani",
    "guj_Gujr": "Gujarati",
    "hin_Deva": "Hindi",
    "kan_Knda": "Kannada",
    "kas_Arab": "Kashmiri",
    "mai_Deva": "Maithili",
    "mal_Mlym": "Malayalam",
    "mar_Deva": "Marathi",
    "mni_Mtei": "Manipuri",
    "npi_Deva": "Nepali",
    "ory_Orya": "Odia",
    "pan_Guru": "Punjabi",
    "san_Deva": "Sanskrit",
    "snd_Deva": "Sindhi",
    "tam_Taml": "Tamil",
    "tel_Telu": "Telugu",
    "urd_Arab": "Urdu",
    "khasi": "Khasi",
    "santhali": "Santali"
}

print("Languages:", len(LANGUAGES))

Languages: 23


In [8]:
def sentence_tokenize(text):
    text = str(text).strip()

    sentences = re.split(r'(?<=[.!?।॥])\s+', text)

    return [
        s.strip()
        for s in sentences
        if len(s.strip()) >= 15
    ]

In [9]:
def collect_sentences(lang_code, n=1000):

    print("Loading:", LANGUAGES[lang_code])

    ds = load_dataset(
        "ai4bharat/IndicCorpV2",
        split=lang_code,
        streaming=True
    )

    sentences = []

    for row in ds:

        for sent in sentence_tokenize(row["text"]):

            sentences.append(sent)

            if len(sentences) == n:
                break

        if len(sentences) == n:
            break

    print("Collected:", len(sentences))

    return sentences

In [10]:
all_texts = []
all_labels = []

for lang in LANGUAGES:

    sentences = collect_sentences(lang, 1000)

    all_texts.extend(sentences)
    all_labels.extend([lang] * len(sentences))

print("\nTotal sentences:", len(all_texts))

Loading: Assamese
Collected: 1000
Loading: Bengali
Collected: 1000
Loading: Bodo
Collected: 1000
Loading: Dogri
Collected: 1000
Loading: Konkani
Collected: 1000
Loading: Gujarati
Collected: 1000
Loading: Hindi
Collected: 1000
Loading: Kannada
Collected: 1000
Loading: Kashmiri
Collected: 1000
Loading: Maithili
Collected: 1000
Loading: Malayalam
Collected: 1000
Loading: Marathi
Collected: 1000
Loading: Manipuri
Collected: 1000
Loading: Nepali
Collected: 1000
Loading: Odia
Collected: 1000
Loading: Punjabi
Collected: 1000
Loading: Sanskrit
Collected: 1000
Loading: Sindhi
Collected: 1000
Loading: Tamil
Collected: 1000
Loading: Telugu
Collected: 1000
Loading: Urdu
Collected: 1000
Loading: Khasi
Collected: 1000
Loading: Santali
Collected: 1000

Total sentences: 23000


In [11]:
train_texts, val_texts, test_texts = [], [], []
train_labels, val_labels, test_labels = [], [], []

for lang in LANGUAGES:

    indices = [
        i for i, x in enumerate(all_labels)
        if x == lang
    ]

    np.random.shuffle(indices)

    for i in indices[:800]:
        train_texts.append(all_texts[i])
        train_labels.append(lang)

    for i in indices[800:900]:
        val_texts.append(all_texts[i])
        val_labels.append(lang)

    for i in indices[900:1000]:
        test_texts.append(all_texts[i])
        test_labels.append(lang)

print("Train:", len(train_texts))
print("Validation:", len(val_texts))
print("Test:", len(test_texts))

Train: 18400
Validation: 2300
Test: 2300


In [12]:
def get_features(text):

    text = text.lower()
    words = re.findall(r'\S+', text)

    features = words.copy()

    # Word bigrams
    features += [
        words[i] + "_" + words[i+1]
        for i in range(len(words)-1)
    ]

    # Character n-grams
    text = re.sub(r'\s+', ' ', text)

    for n in [2, 3, 4]:

        features += [
            text[i:i+n]
            for i in range(len(text)-n+1)
        ]

    return features

In [13]:
MAX_FEATURES = 10000

document_frequency = Counter()

for text in train_texts:

    unique_features = set(get_features(text))

    document_frequency.update(unique_features)

common_features = document_frequency.most_common(
    MAX_FEATURES
)

vocab = {
    feature: i
    for i, (feature, _) in enumerate(common_features)
}

print("Vocabulary size:", len(vocab))

Vocabulary size: 10000


In [14]:
df = np.zeros(
    len(vocab),
    dtype=np.float32
)

for feature, index in vocab.items():
    df[index] = document_frequency[feature]

N = len(train_texts)

print("Training documents:", N)

Training documents: 18400


In [15]:
# Normal IDF

idf_unnormalized = np.log(
    (N + 1) / (df + 1)
) + 1


# Normalized IDF

idf_normalized = (
    idf_unnormalized /
    np.max(idf_unnormalized)
)

print("IDF calculation complete.")

IDF calculation complete.


In [16]:
def calculate_tf(text, method):

    counts = Counter(get_features(text))

    if method == "unnormalized":
        divisor = 1

    elif method == "word_count":
        divisor = len(
            re.findall(r'\S+', text)
        )

    elif method == "max_frequency":
        divisor = max(counts.values())

    tf = {}

    for feature, count in counts.items():

        if feature in vocab:

            index = vocab[feature]

            tf[index] = count / divisor

    return tf

In [17]:
def create_tfidf(texts, tf_method, idf_method):

    if idf_method == "unnormalized":
        idf = idf_unnormalized
    else:
        idf = idf_normalized

    X = np.zeros(
        (len(texts), len(vocab)),
        dtype=np.float32
    )

    for row, text in enumerate(texts):

        tf = calculate_tf(
            text,
            tf_method
        )

        for index, value in tf.items():

            X[row, index] = (
                value * idf[index]
            )

    return X

In [18]:
labels = list(LANGUAGES.keys())

label_to_id = {
    label: i
    for i, label in enumerate(labels)
}

y_train = np.array([
    label_to_id[x]
    for x in train_labels
])

y_val = np.array([
    label_to_id[x]
    for x in val_labels
])

y_test = np.array([
    label_to_id[x]
    for x in test_labels
])

print("Number of classes:", len(labels))

Number of classes: 23


In [19]:
class MyLogisticRegression:

    def __init__(self, lr=0.5, epochs=15):
        self.lr = lr
        self.epochs = epochs

    def softmax(self, z):

        z = z - np.max(
            z,
            axis=1,
            keepdims=True
        )

        e = np.exp(z)

        return e / np.sum(
            e,
            axis=1,
            keepdims=True
        )

    def fit(self, X, y):

        n, d = X.shape
        c = len(np.unique(y))

        self.W = np.zeros(
            (d, c),
            dtype=np.float32
        )

        self.b = np.zeros(
            c,
            dtype=np.float32
        )

        Y = np.zeros(
            (n, c),
            dtype=np.float32
        )

        Y[np.arange(n), y] = 1

        for epoch in range(self.epochs):

            scores = X @ self.W + self.b

            probabilities = self.softmax(scores)

            error = probabilities - Y

            grad_W = (
                X.T @ error
            ) / n

            grad_b = np.mean(
                error,
                axis=0
            )

            self.W -= self.lr * grad_W
            self.b -= self.lr * grad_b

        return self

    def predict(self, X):

        scores = X @ self.W + self.b

        return np.argmax(
            scores,
            axis=1
        )

In [20]:
def macro_f1(y_true, y_pred):

    scores = []

    for c in range(len(labels)):

        tp = np.sum(
            (y_true == c) &
            (y_pred == c)
        )

        fp = np.sum(
            (y_true != c) &
            (y_pred == c)
        )

        fn = np.sum(
            (y_true == c) &
            (y_pred != c)
        )

        precision = tp / (
            tp + fp + 1e-8
        )

        recall = tp / (
            tp + fn + 1e-8
        )

        f1 = (
            2 * precision * recall /
            (precision + recall + 1e-8)
        )

        scores.append(f1)

    return np.mean(scores)

In [21]:
combinations = [
    ("unnormalized", "unnormalized"),
    ("word_count", "unnormalized"),
    ("max_frequency", "unnormalized"),
    ("unnormalized", "normalized"),
    ("word_count", "normalized"),
    ("max_frequency", "normalized")
]

results = []

for tf_method, idf_method in combinations:

    print("\n" + "=" * 50)
    print("TF:", tf_method)
    print("IDF:", idf_method)
    print("=" * 50)

    X_train = create_tfidf(
        train_texts,
        tf_method,
        idf_method
    )

    X_test = create_tfidf(
        test_texts,
        tf_method,
        idf_method
    )

    model = MyLogisticRegression(
        lr=0.5,
        epochs=15
    )

    model.fit(
        X_train,
        y_train
    )

    predictions = model.predict(
        X_test
    )

    accuracy = np.mean(
        predictions == y_test
    )

    f1 = macro_f1(
        y_test,
        predictions
    )

    results.append([
        tf_method,
        idf_method,
        accuracy,
        f1
    ])

    print("Accuracy :", round(accuracy, 4))
    print("Macro-F1 :", round(f1, 4))


TF: unnormalized
IDF: unnormalized
Accuracy : 0.9283
Macro-F1 : 0.9195

TF: word_count
IDF: unnormalized
Accuracy : 0.907
Macro-F1 : 0.8942

TF: max_frequency
IDF: unnormalized
Accuracy : 0.93
Macro-F1 : 0.9286

TF: unnormalized
IDF: normalized
Accuracy : 0.913
Macro-F1 : 0.9079

TF: word_count
IDF: normalized
Accuracy : 0.8965
Macro-F1 : 0.8809

TF: max_frequency
IDF: normalized
Accuracy : 0.8909
Macro-F1 : 0.8773


In [22]:
results_df = pd.DataFrame(
    results,
    columns=[
        "TF Normalization",
        "IDF Normalization",
        "Accuracy",
        "Macro-F1"
    ]
)

results_df

,TF Normalization,IDF Normalization,Accuracy,Macro-F1
0,unnormalized,unnormalized,0.928261,0.919540
1,word_count,unnormalized,0.906957,0.894186
2,max_frequency,unnormalized,0.930000,0.928612
3,unnormalized,normalized,0.913043,0.907865
4,word_count,normalized,0.896522,0.880907
5,max_frequency,normalized,0.890870,0.877313


In [23]:
best = results_df.loc[
    results_df["Macro-F1"].idxmax()
]

print("BEST MODEL")
print("-" * 30)
print("TF       :", best["TF Normalization"])
print("IDF      :", best["IDF Normalization"])
print("Accuracy :", round(best["Accuracy"], 4))
print("Macro-F1 :", round(best["Macro-F1"], 4))

BEST MODEL
------------------------------
TF       : max_frequency
IDF      : unnormalized
Accuracy : 0.93
Macro-F1 : 0.9286


In [24]:
results_df.to_csv(
    "Assignment_2_Results.csv",
    index=False
)

print("Assignment 2 results saved.")

Assignment 2 results saved.


In [25]:
print("=" * 60)
print("ILLM ASSIGNMENT 2 - FINAL RESULTS")
print("=" * 60)

print("Dataset size :", len(all_texts))
print("Languages    :", len(LANGUAGES))
print("Train       :", len(train_texts))
print("Validation  :", len(val_texts))
print("Test        :", len(test_texts))

print("\nAll 6 combinations:")
print(results_df.to_string(index=False))

print("\nBest Macro-F1:",
      round(best["Macro-F1"], 4))

ILLM ASSIGNMENT 2 - FINAL RESULTS
Dataset size : 23000
Languages    : 23
Train       : 18400
Validation  : 2300
Test        : 2300

All 6 combinations:
TF Normalization IDF Normalization  Accuracy  Macro-F1
    unnormalized      unnormalized  0.928261  0.919540
      word_count      unnormalized  0.906957  0.894186
   max_frequency      unnormalized  0.930000  0.928612
    unnormalized        normalized  0.913043  0.907865
      word_count        normalized  0.896522  0.880907
   max_frequency        normalized  0.890870  0.877313

Best Macro-F1: 0.9286
